In [ ]:
# import fitz  # PyMuPDF

# doc = fitz.open("5.pdf")
# full_text = ""
# for page in doc:
#     full_text += page.get_text()

# with open("output.txt", "w", encoding="utf-8") as f:
#     f.write(full_text)

In [ ]:
# import fitz
# import io
# from PIL import Image

# doc = fitz.open("5.pdf")
# for page_index in range(len(doc)):
#     page = doc.load_page(page_index)
#     image_list = page.get_images(full=True)
#     for img_index, img in enumerate(image_list, start=1):
#         xref = img[0]
#         base_image = doc.extract_image(xref)
#         image_bytes = base_image["image"]
#         image_ext = base_image["ext"]
#         with open(f"image_p{page_index+1}_{img_index}.{image_ext}", "wb") as f:
#             f.write(image_bytes)

In [5]:
import fitz
import os
from docx import Document
from docx.shared import Inches, Pt, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH

pdf_path = "5.pdf"              # your input PDF
output_docx = "merged_output.docx"

doc_pdf = fitz.open(pdf_path)
word_doc = Document()

# Page margins
for section in word_doc.sections:
    section.top_margin = Cm(1.5)
    section.bottom_margin = Cm(1.5)
    section.left_margin = Cm(2)
    section.right_margin = Cm(2)

# Default font
style = word_doc.styles["Normal"]
style.font.name = "Calibri"
style.font.size = Pt(11)

os.makedirs("temp_images", exist_ok=True)
img_counter = 0

for page_index in range(len(doc_pdf)):
    page = doc_pdf.load_page(page_index)
    blocks = page.get_text("dict")["blocks"]

    # Sort all blocks by vertical position so text + diagram stay in order
    blocks = sorted(blocks, key=lambda b: b["bbox"][1])

    for block in blocks:
        # TEXT BLOCK
        if block["type"] == 0:
            full_text = ""
            font_size = 11
            is_bold = False

            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    full_text += span["text"] + " "
                    if span.get("size"):
                        font_size = span["size"]
                    if span.get("flags", 0) & 16:
                        is_bold = True

            full_text = full_text.strip()
            if full_text:
                p = word_doc.add_paragraph()
                p.paragraph_format.space_after = Pt(3)
                run = p.add_run(full_text)
                run.font.size = Pt(max(10, min(round(font_size), 14)))
                run.bold = is_bold

        # IMAGE / DIAGRAM BLOCK
        elif block["type"] == 1:
            x0, y0, x1, y1 = block["bbox"]
            width = x1 - x0
            height = y1 - y0

            # Skip tiny images/noise
            if width < 35 or height < 35:
                continue

            clip = fitz.Rect(block["bbox"])
            pix = page.get_pixmap(matrix=fitz.Matrix(2.5, 2.5), clip=clip)

            img_path = f"temp_images/page_{page_index+1}_img_{img_counter}.png"
            pix.save(img_path)
            img_counter += 1

            page_width = page.rect.width
            relative_width = width / page_width
            display_width = min(5.8, max(2.0, relative_width * 6.5))

            p = word_doc.add_paragraph()
            p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            p.paragraph_format.space_before = Pt(4)
            p.paragraph_format.space_after = Pt(4)

            run = p.add_run()
            run.add_picture(img_path, width=Inches(display_width))

    if page_index < len(doc_pdf) - 1:
        word_doc.add_page_break()

word_doc.save(output_docx)
print(f"Saved: {output_docx}")

Saved: merged_output.docx
